# Lab 08: How to build and run an agent with Google ADK

**Notebook 2 of 2: ADK at the Chinook desk**

## Goal

Put ADK to work on a real job: the support desk of the Chinook music shop, where a rep asks about a customer and
the agent looks the facts up in the shop's database. We build the desk agent, keep the rep's conversation in a
file, and then split the desk between two specialists and ask one question that needs both: *How many invoices
does Puja Srivastava have, what do they add up to, and who is her support rep?* With the specialists as
`sub_agents`, the conversation moves: one specialist looks up its part and transfers to the other, the rep gets all
three facts for more than twice the model calls of one agent, and the next question starts at whichever specialist
spoke last. With the same specialists as tools, the conversation stays with a manager.

## What you will learn

- How to build the desk agent from three database lookups, and read its run event by event with its model calls.
- How to keep a conversation in a session stored in a file, and read what the session holds.
- How to make two specialists the `sub_agents` of a front agent, and follow a transfer by reading the author of
  each event.
- How to give the same specialists to a manager as tools with `AgentTool`, and say for each team who holds the
  conversation and what the run costs.

## From the playground to the desk

In notebook 1 we met ADK on everyday questions. Four names carry over:

- `Agent` holds the instruction, the model and the tools.
- A runner takes one message and runs the agent: it calls the model, runs the functions the model asks for, and
  repeats until there is an answer.
- A `Session` is one conversation. The runner loads it before a run and saves each new event as it is produced.
- An `Event` is one step of a run: a function call the model wrote, a function's response, or text.

Now we take it seriously. The tools read a shop's database and a rep will repeat the answers to a customer, so a
missing fact matters. This notebook restates everything it uses, so it runs on its own.

## Setup

This lab has its own environment: choose this folder's `.venv` as the notebook's kernel (see the README in this
folder). The first cell is the same in every lab: it finds the repository folder, reads the model name from
`models.env`, and takes the key from `.env` without printing it. This lab does not use the reasoning effort that
the cell prints.

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


The second cell holds what notebook 1 built up step by step:

- `llm` is the model object. ADK comes from Google and is not tied to Google's models: `OpenAIResponsesLlm` is its
  class for OpenAI models, so the lab runs with the OpenAI key alone.
- `reply_text(event)` returns the text of an event. An event's content is a list of parts; we join the text parts
  and leave out any thought part (the model's reasoning, which arrives unreadable).
- `ask(runner, session, text)` sends one user message and returns the list of events of the run. `runner.run_async`
  hands over one event per step, and `async for` waits for each of them. A message is a `types.Content`: a `role`
  and a list of `parts`. Ours has one text part.

In [2]:
from google.adk.agents import Agent
from google.adk.integrations.openai import OpenAIResponsesLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

llm = OpenAIResponsesLlm(model=MODEL)


def reply_text(event):
    """The text of an event: its text parts joined, leaving out the model's thoughts."""
    return "".join(part.text for part in event.content.parts if part.text and not part.thought)


async def ask(runner, session, text):
    """Send one user message in a session and return the events of the run."""
    message = types.Content(role="user", parts=[types.Part(text=text)])
    events = []
    async for event in runner.run_async(user_id=session.user_id, session_id=session.id, new_message=message):
        events.append(event)
    return events

---

## Part A: Build the desk agent from three database lookups

The shop's data is `data/chinook.db`, a SQLite file with the customers, their invoices and the support reps who
look after them. A rep at the desk asks things like:

> How many invoices does Helena Holý have, what do they add up to, and who is her support rep?

The question asks for three facts, and they are in three tables. We write one small function for each lookup and
call them ourselves first, so we know the right answers before any model is involved.

### The three lookups

- `find_customer(name)` finds a customer by full name and returns their `customer_id`.
- `get_invoices(customer_id)` counts that customer's invoices and adds up their total.
- `get_support_rep(customer_id)` names the rep who looks after them.

One detail in the first lines: `mode=ro` opens the database **read-only**, so no tool can change the shop's data.
(`uri=True` is what lets SQLite read that option from the file name.)

Each function has type hints and a docstring. From these, ADK writes the description that the model reads: the
function's name, its docstring, and the type of each argument.

In [3]:
import sqlite3

DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}

We look up two customers by hand: Helena, and Puja Srivastava, the customer in Part C's question.

In [4]:
for name in ["Helena Holý", "Puja Srivastava"]:
    customer = find_customer(name)[0]
    print(customer)
    print("   ", get_invoices(customer["customer_id"]))
    print("   ", get_support_rep(customer["customer_id"]))

{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}
    {'customer_id': 6, 'invoice_count': 7, 'total': 49.62}
    {'customer_id': 6, 'support_rep': 'Steve Johnson'}
{'customer_id': 59, 'name': 'Puja Srivastava', 'country': 'India'}
    {'customer_id': 59, 'invoice_count': 6, 'total': 36.64}
    {'customer_id': 59, 'support_rep': 'Jane Peacock'}


These are the facts every answer in this notebook is checked against:

| Customer | `customer_id` | Invoices | Total | Support rep |
|---|---|---|---|---|
| Helena Holý | 6 | 7 | \$49.62 | Steve Johnson |
| Puja Srivastava | 59 | 6 | \$36.64 | Jane Peacock |

**Notice** the order the lookups need. `get_invoices` and `get_support_rep` take a `customer_id`, and only
`find_customer` can supply it. Nothing in the agent's instruction says so; the model has to choose that order
from the descriptions of the three functions.

### The desk agent

The agent gets the three functions in `tools`. `InMemoryRunner` keeps its sessions in this Python process, which
is enough for one question. `app_name` is a label: ADK stores every conversation under an application name and a
user, and `user_id="rep"` says whose conversation it is.

`await` runs an `async` function and waits for its result. A notebook lets us write it directly in a cell.

In [5]:
desk = Agent(
    name="desk",
    model=llm,
    instruction=("You help the support reps of a music shop. Look up every fact with the tools; "
                 "never answer from memory. Answer in one or two sentences."),
    tools=[find_customer, get_invoices, get_support_rep],
)

desk_runner = InMemoryRunner(agent=desk, app_name="chinook_desk")
session = await desk_runner.session_service.create_session(app_name="chinook_desk", user_id="rep")

HELENA = "How many invoices does Helena Holý have, what do they add up to, and who is her support rep?"

events = await ask(desk_runner, session, HELENA)

print(reply_text(events[-1]))

Helena Holý has 7 invoices totaling $49.62. Her support rep is Steve Johnson.


Seven invoices, \$49.62, Steve Johnson: the answer agrees with the table.

### Read the run event by event

A right answer does not tell us how it was reached. The events do. `show_run` prints what each event holds:

- `event.get_function_calls()` returns the function calls the model wrote in this event: a name and arguments.
- `event.get_function_responses()` returns what our functions gave back.
- `reply_text(event)` is the event's text, if it has any.

Every line starts with the event's number and its `author`, the agent that wrote the event.

In [6]:
def show_run(events):
    """Print what each event holds, with its number and its author in front."""
    for number, event in enumerate(events, start=1):
        for call in event.get_function_calls():
            print(f"{number:>2} {event.author:12} asks for   {call.name} {call.args}")
        for response in event.get_function_responses():
            print(f"{number:>2} {event.author:12} gets back  {response.name} {response.response}")
        if reply_text(event):
            print(f"{number:>2} {event.author:12} says       {reply_text(event)}")


show_run(events)

 1 desk         asks for   find_customer {'name': 'Helena Holý'}
 2 desk         gets back  find_customer {'result': [{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}]}
 3 desk         asks for   get_invoices {'customer_id': 6}
 3 desk         asks for   get_support_rep {'customer_id': 6}
 4 desk         gets back  get_invoices {'customer_id': 6, 'invoice_count': 7, 'total': 49.62}
 4 desk         gets back  get_support_rep {'customer_id': 6, 'support_rep': 'Steve Johnson'}
 5 desk         says       Helena Holý has 7 invoices totaling $49.62. Her support rep is Steve Johnson.


Five events:

1. The model asked for `find_customer` first. The other two lookups need a `customer_id`, and only `find_customer`
   returns one.
2. ADK ran the function and put what it returned in a function response. `find_customer` returns a list, so ADK
   wrapped it in a dictionary under the key `result`. ADK stores a function's response under the agent that asked
   for it, so the author of this event is `desk` too.
3. Once it had the id, the model asked for `get_invoices` and `get_support_rep` in one event. Neither depends on
   the other.
4. Both results came back in one event. These two functions return dictionaries, which ADK passes on as they are.
5. The model wrote the answer.

### Count the model calls

An event that came from the model carries `usage_metadata`, the token counts of that model call. An event our
function produced has none. `model_calls` keeps the input tokens of each model call, so its length is the number
of times the model was called.

In [7]:
def model_calls(events):
    """The input tokens of each model call in a list of events."""
    return [event.usage_metadata.prompt_token_count for event in events if event.usage_metadata]


print("events:     ", len(events))
print("model calls:", len(model_calls(events)))
print("input tokens per model call:", model_calls(events))

events:      5
model calls: 3
input tokens per model call: [221, 281, 392]


Five events, three model calls. Events 1, 3 and 5 came from the model; events 2 and 4 came from our functions. The
input tokens grow from 221 to 281 to 392, because every model call carries the conversation so far, with the
function calls and responses before it. The model keeps nothing between calls, so the whole conversation is sent
again each time, and we pay for it each time.

**Checkpoint A.** The lookups give Helena 7 invoices, \$49.62 and Steve Johnson, and Puja 6, \$36.64 and Jane
Peacock. The desk agent answers Helena's question with her three facts. `find_customer` is the first function
call, and the input tokens grow from one model call to the next.

---

## Part B: Keep the rep's conversation in a file

A rep asks a question, then a follow-up that depends on it: *and how much has she spent with us?* A session keeps
the conversation, so the follow-up works. But the sessions of `InMemoryRunner` are kept in the memory of this Python process. When
the program stops, they are gone, and a desk application restarts every time it is updated.

To keep sessions after the program stops, we change the session service. `InMemoryRunner` is a `Runner` that comes with an `InMemorySessionService`. We
build the `Runner` ourselves and give it a service that writes to a file:

- `SqliteSessionService(db_path=...)` stores sessions in a SQLite file. Ours goes in `outputs/` beside this
  notebook, a folder git ignores.
- `Runner(agent=..., app_name=..., session_service=...)` is the same runner as before, with the service we chose.
- `session_id="rep-helena"` gives the session an id we can find again. If we leave it out, ADK makes a random one.

`unlink(missing_ok=True)` deletes the file first, so running this cell again starts fresh.

In [8]:
from google.adk.runners import Runner
from google.adk.sessions.sqlite_session_service import SqliteSessionService

SESSION_FILE = Path("outputs") / "sessions.db"
SESSION_FILE.parent.mkdir(exist_ok=True)
SESSION_FILE.unlink(missing_ok=True)

service = SqliteSessionService(db_path=str(SESSION_FILE))
runner = Runner(agent=desk, app_name="chinook_desk", session_service=service)

session = await service.create_session(app_name="chinook_desk", user_id="rep", session_id="rep-helena")

print("session:", session.id, "| events:", len(session.events))

session: rep-helena | events: 0


The session exists and is empty. Now two questions in it. The second one names nobody.

In [9]:
events = await ask(runner, session, "Which country is Helena Holý from?")
print("run 1:", reply_text(events[-1]))

events = await ask(runner, session, "And how much has she spent with us?")
print("run 2:", reply_text(events[-1]))

run 1: Helena Holý is from the Czech Republic.


run 2: Helena Holý has spent **$49.62** with you.


Run 2 answers with \$49.62. The question said only *she*. The name came from the earlier events in the session.

### What the file holds

A restarted program has none of our variables. It has the file. We imitate a restarted program: we make a second
service object on the same file, ask it for the session by its id, and print the stored events with `show_run`.

In [10]:
reopened = SqliteSessionService(db_path=str(SESSION_FILE))
stored = await reopened.get_session(app_name="chinook_desk", user_id="rep", session_id="rep-helena")

print("events stored:", len(stored.events))
show_run(stored.events)

events stored: 8
 1 user         says       Which country is Helena Holý from?
 2 desk         asks for   find_customer {'name': 'Helena Holý'}
 3 desk         gets back  find_customer {'result': [{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}]}
 4 desk         says       Helena Holý is from the Czech Republic.
 5 user         says       And how much has she spent with us?
 6 desk         asks for   get_invoices {'customer_id': 6}
 7 desk         gets back  get_invoices {'customer_id': 6, 'invoice_count': 7, 'total': 49.62}
 8 desk         says       Helena Holý has spent **$49.62** with you.


Eight events, both runs in order: the first question, its `find_customer` call and response, and the answer; then
the follow-up, one lookup with its response, and its answer. The variables `service` and `session` were not used
in this read. The second service object got everything from the file.

When run 2 started, the runner loaded events 1 to 4 and sent them in front of the follow-up. That is how the model
could read *she* as Helena Holý. It also explains event 6: the model asked for `get_invoices` with `customer_id` 6
at once, without looking the customer up again, because the id was already in event 3.

**Important:** what a session stores has a cost. Everything stored is sent to the model again on every run, so a long
conversation costs more with every new question.

A different id is a different conversation. We ask the same file for a session nobody created:

In [11]:
print("under rep-puja:", await reopened.get_session(app_name="chinook_desk", user_id="rep", session_id="rep-puja"))

under rep-puja: None


`None`: the file holds no session under that id.

**Checkpoint B.** The follow-up is answered with \$49.62 although it names nobody. A second service object on the
same file finds the session with both runs' events, and finds nothing under another id.

---

## Part C: Split the desk between two specialists and follow the transfers

One agent with three tools works well. A real desk has many more tools, and one agent with thirty tools and a page
of instructions starts choosing the wrong one. A common answer is to split the work: small agents, each with its
own instruction and tools, and one agent in front.

The desk gets questions that belong to more than one specialist:

> How many invoices does Puja Srivastava have, what do they add up to, and who is her support rep?

Part of this question is about invoices and part is about support reps. The right answer, from Part A:
**6 invoices, \$36.64, Jane Peacock.**

### The baseline: one agent with all three tools

In [12]:
PUJA = "How many invoices does Puja Srivastava have, what do they add up to, and who is her support rep?"

session = await desk_runner.session_service.create_session(app_name="chinook_desk", user_id="rep")
events = await ask(desk_runner, session, PUJA)

print(reply_text(events[-1]))
print("model calls:", len(model_calls(events)))

Puja Srivastava has 6 invoices totaling $36.64. Her support rep is Jane Peacock.
model calls: 3


All three facts in 3 model calls. This is the number the two teams are compared with.

### Two specialists

Each specialist is an ordinary `Agent` with only the tools for its part:

- `invoices` can find a customer and read their invoices.
- `support_reps` can find a customer and name their rep.

Neither can do the other's job. `description` is one sentence about the agent. Other agents read it when they
decide who should take a question.

In [13]:
invoices = Agent(
    name="invoices",
    model=llm,
    description="Answers questions about a customer's invoices: how many, and their total.",
    instruction=("You answer questions about a customer's invoices. Look up every fact with your tools; "
                 "never answer from memory. Answer in one or two sentences."),
    tools=[find_customer, get_invoices],
)

support_reps = Agent(
    name="support_reps",
    model=llm,
    description="Answers who a customer's support rep is.",
    instruction=("You answer who looks after a customer. Look up every fact with your tools; "
                 "never answer from memory. Answer in one or two sentences."),
    tools=[find_customer, get_support_rep],
)

print(invoices.name, [tool.__name__ for tool in invoices.tools])
print(support_reps.name, [tool.__name__ for tool in support_reps.tools])

invoices ['find_customer', 'get_invoices']
support_reps ['find_customer', 'get_support_rep']


### A team: the specialists as `sub_agents`

The front agent, `triage`, has no tools of its own. It has `sub_agents`: the agents it may pass a conversation to.
ADK calls passing a conversation a **transfer**. It gives the model one extra function, `transfer_to_agent`, and
when the model calls it with an agent's name, that agent takes over: it reads the conversation so far and writes
the next events.

In [14]:
triage = Agent(
    name="triage",
    model=llm,
    instruction=("You route each request to the right specialist. Transfer to invoices for invoice counts and "
                 "totals, and to support_reps for who looks after a customer."),
    sub_agents=[invoices, support_reps],
)

print("sub-agents of triage:", [agent.name for agent in triage.sub_agents])
print("parent of invoices:  ", invoices.parent_agent.name)

sub-agents of triage: ['invoices', 'support_reps']
parent of invoices:   triage


`triage` has two sub-agents, and each of them now has `triage` as its parent.

Before you run the next cell: the question needs both specialists, and a transfer goes to one. Which agents will
write events, and will the answer hold all three facts?

In the output, watch the second column. It is the author of each event.

In [15]:
team_runner = InMemoryRunner(agent=triage, app_name="chinook_team")
team_session = await team_runner.session_service.create_session(app_name="chinook_team", user_id="rep")

events = await ask(team_runner, team_session, PUJA)

show_run(events)
print("model calls:", len(model_calls(events)))

App "chinook_team" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


 1 triage       asks for   transfer_to_agent {'agent_name': 'invoices'}
 1 triage       asks for   transfer_to_agent {'agent_name': 'support_reps'}
 2 triage       gets back  transfer_to_agent {'result': None}
 2 triage       gets back  transfer_to_agent {'result': None}
 3 support_reps asks for   find_customer {'name': 'Puja Srivastava'}
 4 support_reps gets back  find_customer {'result': [{'customer_id': 59, 'name': 'Puja Srivastava', 'country': 'India'}]}
 5 support_reps asks for   get_support_rep {'customer_id': 59}
 6 support_reps gets back  get_support_rep {'customer_id': 59, 'support_rep': 'Jane Peacock'}
 7 support_reps asks for   transfer_to_agent {'agent_name': 'invoices'}
 8 support_reps gets back  transfer_to_agent {'result': None}
 9 invoices     asks for   find_customer {'name': 'Puja Srivastava'}
10 invoices     gets back  find_customer {'result': [{'customer_id': 59, 'name': 'Puja Srivastava', 'country': 'India'}]}
11 invoices     asks for   get_invoices {'customer_id':

All three facts are in event 15: 6 invoices, \$36.64, Jane Peacock. Follow the second column to see how they
got there:

- **Events 1 and 2: `triage` asked for two transfers at once,** to `invoices` and to `support_reps`. A
  conversation can go to one agent only, and ADK takes the transfer named last: `support_reps`.
  `transfer_to_agent` returns nothing, so its response is `{'result': None}`.
- **Events 3 to 6: `support_reps` did its part.** It found the customer and looked up her rep.
- **Events 7 and 8: `support_reps` transferred the conversation to `invoices`.** Our instructions did not ask for
  that. In ADK a sub-agent may transfer to the other sub-agents of its parent, and back to the parent, unless we
  switch that off.
- **Events 9 to 12: `invoices` did its part.** It looked the customer up for itself and read her invoices.
- **Events 13 to 15: `invoices` transferred the conversation back, and `support_reps` wrote the answer.** By then
  every fact was in the conversation, and `support_reps` put them together.

**Key lesson:** in an ADK team the conversation moves, and the agent that holds it writes the next event. That is
why the answer is complete here: neither specialist can look up all three facts, but together they looked up all
three. The cost is in the last line of the output: 8 model calls against 3 for one agent. Each specialist looked
the customer up for itself (two `find_customer` calls where one agent made one), and each of the three transfers
took a model call that looked nothing up. On other runs of ours the count was between 7 and 11, depending on how
often the specialists passed the conversation back and forth.

The first lines of the output, before event 1, are a warning from ADK, not an error. It says that every transfer
changes the instruction and the tools at the start of the request, so a prompt cache cannot be reused after a
transfer.

### Where the next question goes

The rep's conversation continues. Before you run it: which agent will write the first event?

In [16]:
events = await ask(team_runner, team_session, "And who is Helena Holý's support rep?")

show_run(events)

 1 support_reps asks for   find_customer {'name': 'Helena Holý'}
 2 support_reps gets back  find_customer {'result': [{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}]}
 3 support_reps asks for   get_support_rep {'customer_id': 6}
 4 support_reps gets back  get_support_rep {'customer_id': 6, 'support_rep': 'Steve Johnson'}
 5 support_reps says       Helena Holý’s support rep is Steve Johnson.


The first event was written by `support_reps`, not by `triage`. ADK gave the new message to the agent that spoke
last in this session. This time that agent had the right tool, so it looked the rep up and answered: Steve Johnson.

**Important:** after a transfer, `triage` is no longer in front. Had the question been about invoices,
`support_reps` would still have received it, and it would have reached the invoices specialist only if
`support_reps` passed the conversation on.

### The specialists as tools: the manager keeps the conversation

There is a second way to connect the same specialists. `AgentTool` wraps a whole agent as a tool. When the manager
calls it, ADK runs that specialist on a request the manager writes and returns the specialist's answer as the
function response. The manager then carries on, as it would after any function.

`invoices` and `support_reps` are still sub-agents of `triage`, so inside a tool call they would keep their
`transfer_to_agent` function. `clone()` makes a copy with no parent, which has only its two lookups.

In [17]:
from google.adk.tools import AgentTool

manager = Agent(
    name="manager",
    model=llm,
    instruction=("You help the support reps of a music shop. Ask the specialists for every fact; never answer from "
                 "memory. Answer in one or two sentences."),
    tools=[AgentTool(agent=invoices.clone()), AgentTool(agent=support_reps.clone())],
)

manager_runner = InMemoryRunner(agent=manager, app_name="chinook_manager")
manager_session = await manager_runner.session_service.create_session(app_name="chinook_manager", user_id="rep")

events = await ask(manager_runner, manager_session, PUJA)

show_run(events)
print("model calls in this list:", len(model_calls(events)))

 1 manager      asks for   invoices {'request': 'How many invoices does customer Puja Srivastava have, and what is their total amount?'}
 1 manager      asks for   support_reps {'request': 'Who is the support rep for customer Puja Srivastava?'}
 2 manager      gets back  invoices {'result': 'Puja Srivastava has 6 invoices, totaling $36.64.'}
 2 manager      gets back  support_reps {'result': 'Puja Srivastava’s support rep is Jane Peacock.'}
 3 manager      says       Puja Srivastava has 6 invoices totaling $36.64, and her support rep is Jane Peacock.
model calls in this list: 2


Three events, and `manager` wrote every one of them:

1. The manager asked both specialists in one event. Each got a `request` the manager wrote for it (*Who is the
   support rep for customer Puja Srivastava?*), not the rep's original question.
2. Both answers came back as function responses.
3. The manager put them together: all three facts.

The specialists' own lookups are not in this list. Each specialist ran its own loop inside the tool call, with its
own model calls, and only its final answer came back. So the 2 printed above counts the manager's model calls
only. The specialists' model calls are extra. Each specialist needs at least three for its part (find the customer,
look up the fact, answer), so this run cost at least 2 + 3 + 3 = 8 model calls: no cheaper than the team.

The same follow-up as before. Which agent will write the first event this time?

In [18]:
events = await ask(manager_runner, manager_session, "And who is Helena Holý's support rep?")

show_run(events)

 1 manager      asks for   support_reps {'request': 'Who is the support rep for customer Helena Holý?'}
 2 manager      gets back  support_reps {'result': 'Helena Holý’s support rep is Steve Johnson.'}
 3 manager      says       Helena Holý’s support rep is Steve Johnson.


`manager` wrote the first event. It asked `support_reps` and answered: Steve Johnson. The manager never gave the
conversation away, so the next question came to it.

### Who holds the conversation

| | One agent, three tools | Specialists as `sub_agents` | Specialists as tools (`AgentTool`) |
|---|---|---|---|
| Who decides the next step | the agent's model | whichever agent holds the conversation | the manager's model |
| Does the front agent get the conversation back | it never leaves | only if a specialist transfers it back | yes, after every specialist |
| Who takes the next question | the agent | the agent that spoke last | the manager |
| Puja's question in our run | three facts, 3 model calls | three facts, 8 model calls, 3 transfers | three facts, 2 model calls by the manager and at least 6 inside the specialists |
| Good for | tools that fit in one prompt: start here | a conversation that should stay with a specialist once it gets there | a question with several parts that needs one combined answer |

**Checkpoint C.** One agent answers Puja's three facts. The team with `sub_agents` shows more than one author in
the second column, with `transfer_to_agent` calls between them, and the follow-up starts at the agent that spoke
last. With `AgentTool` every event is written by `manager`, and the follow-up starts at `manager`.

---

## In production

- **Sessions in a database server.** `SqliteSessionService` suits one machine. For an application on several
  servers ADK has `DatabaseSessionService`, which needs the extra `google-adk[db]`.
- **Transfers and the prompt cache.** ADK warned us in Part C: every transfer changes the instruction and the tools
  at the start of the request, so the provider's prompt cache (a lower price for the repeated start of a request) starts again after each one. A long conversation with many
  transfers pays the full input price more often.
- **Writes need approval.** These tools only read. A tool that changes something (a refund) should stop for a
  person's approval before it runs.

## What you can do now

- Build an agent over a database from plain lookup functions, and read its run event by event with its model calls
  (Part A).
- Keep a conversation in a file with `SqliteSessionService`, and read what the session holds (Part B).
- Connect specialists as `sub_agents` and follow the transfers by reading the author of each event (Part C).
- Connect the same specialists as tools with `AgentTool`, and say who holds the conversation in each team and
  what the run costs (Part C).

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'google.adk'` | the notebook's kernel is not this lab's environment | choose `08_Google_ADK/.venv` as the kernel (see the README in this folder) |
| `SessionNotFoundError: Session not found: rep-puja` | the runner was given a `session_id` that its session service does not hold | create the session with `create_session` first, with the same `app_name` and `user_id` |
| a follow-up question answered as if it were the first | the second `ask` got another session | pass the same `session` to every `ask` of the conversation |
| `sqlite3.OperationalError: unable to open database file` | `data/chinook.db` is missing from the repository folder | restore the file (clone the repository again, or `git checkout data/chinook.db`) |

## Challenge

1. **Make the transfers one way.** An ADK agent has two switches, `disallow_transfer_to_parent` and
   `disallow_transfer_to_peers`. `clone(update=...)` takes a dictionary of the fields to change in
   the copy: `invoices.clone(update={"disallow_transfer_to_parent": True, "disallow_transfer_to_peers": True})`.
   Make such a copy of each specialist, give them to a new triage agent as `sub_agents`, and ask Puja's question in a
   new session. Print the run with `show_run` and the number of model calls. Expected: the conversation
   goes to one specialist and stays there. The reply has that specialist's facts and says it cannot look up the
   rest. In our run the reply named the support rep and had no invoice facts, in 4 model calls.
2. **Carry a conversation through the manager, in a file.** Give the manager a `Runner` with a
   `SqliteSessionService` (a file under `outputs/`), ask Puja's question, then *And what about Helena Holý?*
   Expected: the second answer has 7 invoices, \$49.62 and Steve Johnson, and the `request` the manager wrote for each
   specialist names Helena Holý in full, because a specialist sees only that request, never the session.

The worked solution, run with its outputs, is in `solved/08b_google_adk_at_the_desk_solution.ipynb`.

## Resources

- [Agent Development Kit](https://adk.dev/): the documentation's home
- [LLM agents](https://adk.dev/agents/llm-agents/): `Agent`, its instruction, its tools and its transfer settings
- [Collaborative agent workflows](https://adk.dev/workflows/collaboration/): `sub_agents` and transfers
- [Sessions](https://adk.dev/sessions/session/): what a session holds, and the session services
- [Events](https://adk.dev/events/): what an event carries and how to read it
- [Function tools](https://adk.dev/tools-custom/function-tools/): Python functions as tools, and `AgentTool`

Back to [notebook 1](08a_meet_google_adk.ipynb). Next:
[Lab 09: How to compose LangChain's building blocks](../09_LangChain/README.md).